# 06 — Sources, Authors, Institutions, Countries, and Collaboration

Calcula actores bibliométricos, producción, citas y redes de colaboración por autor y país.

> Notebook reproducible del Paper Semantic Bibliometric Toolkit v1.0.0. No elimina documentos ni convierte resultados exploratorios en conclusiones sin revisión manual.

## 1. Cargar corpus

In [ ]:

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
current = Path.cwd().resolve()
for candidate in [current] + list(current.parents):
    if (candidate / "config.json").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró config.json.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core import (
    find_project_root, read_json, load_config, ensure_directories,
    load_scopus_csv, load_clean_corpus, normalize_spacing,
    normalize_identifier, normalize_title, normalize_keyword,
    split_semicolon, frequency_table, save_csv, save_json,
    markdown_table, combined_text
)

CONFIG = load_config(PROJECT_ROOT)
ensure_directories(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
THESAURUS_DIR = PROJECT_ROOT / "thesaurus"
VALIDATION_DIR = PROJECT_ROOT / "validation"

print("Project root:", PROJECT_ROOT)
print("Toolkit version:", CONFIG.get("toolkit_version"))
print("Configured corpus:", PROJECT_ROOT / CONFIG["input_csv"])

df = load_clean_corpus(PROJECT_ROOT)
from parsing import parse_author_full_names, parse_affiliations, extract_country_from_affiliation, extract_institution_candidate, build_pair_edges


## 2. Fuentes

In [ ]:

df["Cited by numeric"] = pd.to_numeric(df["Cited by"], errors="coerce").fillna(0)
sources = df.groupby("Source title normalized").agg(Documents=("EID", "nunique"), Citations=("Cited by numeric", "sum")).reset_index().sort_values(["Documents", "Citations"], ascending=[False, False])
save_csv(sources, OUTPUT_DIR / "06_source_statistics.csv")
display(sources.head(30))


## 3. Autores y colaboración

In [ ]:

author_rows = []
document_authors = {}
for _, record in df.iterrows():
    parsed = parse_author_full_names(record.get("Author full names", ""))
    names = []
    for author in parsed:
        name = author["Author full name"]
        author_rows.append({"EID": record["EID"], "Year": record["Year"], "Author": name, "Author ID": author["Author ID"], "Cited by": record["Cited by numeric"]})
        names.append(author["Author ID"] or name)
    document_authors[record["EID"]] = names
authors = pd.DataFrame(author_rows)
author_stats = authors.drop_duplicates(["EID", "Author ID", "Author"]).groupby(["Author", "Author ID"]).agg(Documents=("EID", "nunique"), Citations=("Cited by", "sum")).reset_index().sort_values(["Documents", "Citations"], ascending=[False, False])
author_edges = build_pair_edges(document_authors, "Source", "Target")
save_csv(authors, OUTPUT_DIR / "06_authors_exploded.csv")
save_csv(author_stats, OUTPUT_DIR / "06_author_statistics.csv")
save_csv(author_edges, OUTPUT_DIR / "06_author_collaboration_edges.csv")
display(author_stats.head(30))


## 4. Instituciones, países y colaboración

In [ ]:

aff_rows = []
document_countries = {}
for _, record in df.iterrows():
    countries = []
    for affiliation in parse_affiliations(record.get("Affiliations", "")):
        country = extract_country_from_affiliation(affiliation)
        institution = extract_institution_candidate(affiliation)
        aff_rows.append({"EID": record["EID"], "Year": record["Year"], "Affiliation": affiliation, "Institution": institution, "Country": country, "Cited by": record["Cited by numeric"]})
        if country:
            countries.append(country)
    document_countries[record["EID"]] = countries
affiliations = pd.DataFrame(aff_rows)
institution_stats = affiliations[affiliations["Institution"].ne("")].drop_duplicates(["EID", "Institution"]).groupby("Institution").agg(Documents=("EID", "nunique"), Citations=("Cited by", "sum")).reset_index().sort_values(["Documents", "Citations"], ascending=[False, False])
country_stats = affiliations[affiliations["Country"].ne("")].drop_duplicates(["EID", "Country"]).groupby("Country").agg(Documents=("EID", "nunique"), Citations=("Cited by", "sum")).reset_index().sort_values(["Documents", "Citations"], ascending=[False, False])
country_edges = build_pair_edges(document_countries, "Source", "Target")
save_csv(affiliations, OUTPUT_DIR / "06_affiliations_countries.csv")
save_csv(institution_stats, OUTPUT_DIR / "06_institution_statistics.csv")
save_csv(country_stats, OUTPUT_DIR / "06_country_statistics.csv")
save_csv(country_edges, OUTPUT_DIR / "06_country_collaboration_edges.csv")
display(country_stats.head(30))


## 5. Cobertura y metodología

In [ ]:

country_coverage = round(affiliations["Country"].ne("").sum() / max(1, len(affiliations)) * 100, 2)
summary = {"Sources": len(sources), "Authors": len(author_stats), "Institutions": len(institution_stats), "Countries": len(country_stats), "Country extraction coverage (%)": country_coverage}
save_json(summary, OUTPUT_DIR / "06_actor_analysis_summary.json")
text = (
    "Authors were disambiguated using Scopus Author IDs whenever available. Countries were extracted from affiliation endings and "
    "institution names were treated as auditable candidates rather than automatically canonical entities. Collaboration edges represent "
    "co-occurrence within the same publication."
)
(REPORTS_DIR / "06_actor_methodology.txt").write_text(text, encoding="utf-8")
print(text)
